In [3]:
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma
from pdf_chunk import text_spliter
from models import get_model , get_embeddings

In [4]:
model = get_model()
embedding_model = get_embeddings()
chunks = text_spliter

HTTP Error 503 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 8s [Retry 4/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 8s [Retry 5/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Loading weights: 100%|████

In [19]:
chroma_vectorstore=Chroma.from_documents(chunks ,embedding_model )
chroma_retriever = chroma_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

In [20]:
faiss_vectorstore = FAISS.from_documents(chunks,embedding_model)
faiss_retriever = faiss_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

In [16]:
from langchain_classic.retrievers import EnsembleRetriever

In [21]:

lotr =EnsembleRetriever(retrievers=[chroma_retriever,faiss_retriever],weights=[0.5, 0.5],)

In [29]:
docs = []
for i, chunk in enumerate(lotr.invoke("current university")):
    print(f"Chunk {i+1}:")
    docs.append(chunk)
    print(chunk.page_content)
    print("-" * 50)

Chunk 1:
Role: MLOps Engineer
Company: Lyceum Infotech Private Limited
Duration: September 2022 to December 2024
Location: Pune, India
--------------------------------------------------
Chunk 2:
Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Secondary Certificate): 57%


COMPLETE TECHNICAL SKILLS
--------------------------------------------------
Chunk 3:
Role: Assistant Professor
Institution: Guru Gobind Singh Foundation
Duration: May 2019 to August 2022
Location: Nashik, India
--------------------------------------------------
Chunk 4:
ENGINEERING PHILOSOPHY
--------------------------------------------------
Chunk 5:
CAREER OBJECTIVE
--------------------------------------------------
Chunk 6:
Beyond LLM systems, Rohanta has a strong foundation in classical machine learning and deep learning. He has implemented text classification systems using logistic regression and support vector machines. He understands model evaluation metrics including precision, recall

In [30]:
#  Removes duplicate/similar chunks based on embedding similarity
from langchain_community.document_transformers import EmbeddingsRedundantFilter
redundant_filter = EmbeddingsRedundantFilter(embeddings=embedding_model)

In [39]:
# Remove duplicates
filtered_docs = redundant_filter.transform_documents(docs)
a = 0
for doc in filtered_docs:
    a += 1
    print(a)
    print(doc.page_content)
    print("-" * 50)

1
Role: MLOps Engineer
Company: Lyceum Infotech Private Limited
Duration: September 2022 to December 2024
Location: Pune, India
--------------------------------------------------
2
Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Secondary Certificate): 57%


COMPLETE TECHNICAL SKILLS
--------------------------------------------------
3
Role: Assistant Professor
Institution: Guru Gobind Singh Foundation
Duration: May 2019 to August 2022
Location: Nashik, India
--------------------------------------------------
4
ENGINEERING PHILOSOPHY
--------------------------------------------------
5
CAREER OBJECTIVE
--------------------------------------------------
6
Beyond LLM systems, Rohanta has a strong foundation in classical machine learning and deep learning. He has implemented text classification systems using logistic regression and support vector machines. He understands model evaluation metrics including precision, recall, F1-score, and ROC-AUC, and applies cross

In [36]:
from langchain_community.document_transformers import EmbeddingsClusteringFilter

In [37]:
# Groups docs into clusters and picks the most representative from each
clustering_filter = EmbeddingsClusteringFilter(
    embeddings=embedding_model,
    num_clusters=4,        # number of clusters to form
    num_nearest=1          # docs to pick per cluster
)

In [41]:
filtered_docs = clustering_filter.transform_documents(filtered_docs)
a = 0
for doc in filtered_docs:
    a += 1
    print(a)
    print(doc.page_content)
    print("-" * 50)

1
Role: MLOps Engineer
Company: Lyceum Infotech Private Limited
Duration: September 2022 to December 2024
Location: Pune, India
--------------------------------------------------
2
Degree: Master of Engineering (ME)
Institution: Savitribai Phule Pune University, Pune, India
Duration: 2017 to 2020
Grade: First Class with Distinction, CGPA 8
Thesis: Sentiment Analysis on Customer Reviews. Built NLP pipelines using TF-IDF, classical ML models, and LSTM networks. Deployed via Flask API.

Degree: Bachelor of Engineering (BE)
Institution: Savitribai Phule Pune University, Pune, India
Duration: 2013 to 2017
Grade: First Class with Distinction, 70%
--------------------------------------------------
3
Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Secondary Certificate): 57%


COMPLETE TECHNICAL SKILLS
--------------------------------------------------
4
ENGINEERING PHILOSOPHY
--------------------------------------------------


In [42]:
from langchain_community.document_transformers import LongContextReorder

In [54]:
# Based on the "Lost in the Middle" research paper — LLMs tend to forget context placed in the middle of a long prompt. 
# This reorders docs so that the most relevant chunks are placed at the beginning and end, and least relevant in the middle
reorder = LongContextReorder()
reordered_docs = reorder.transform_documents(filtered_docs)
a = 0
contents = []
for doc in reordered_docs:
    a += 1
    print(a)
    contents.append(doc.page_content)
    print(doc.page_content)
    print("---")

1
Degree: Master of Engineering (ME)
Institution: Savitribai Phule Pune University, Pune, India
Duration: 2017 to 2020
Grade: First Class with Distinction, CGPA 8
Thesis: Sentiment Analysis on Customer Reviews. Built NLP pipelines using TF-IDF, classical ML models, and LSTM networks. Deployed via Flask API.

Degree: Bachelor of Engineering (BE)
Institution: Savitribai Phule Pune University, Pune, India
Duration: 2013 to 2017
Grade: First Class with Distinction, 70%
---
2
ENGINEERING PHILOSOPHY
---
3
Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Secondary Certificate): 57%


COMPLETE TECHNICAL SKILLS
---
4
Role: MLOps Engineer
Company: Lyceum Infotech Private Limited
Duration: September 2022 to December 2024
Location: Pune, India
---


In [47]:
# Query
#   ↓
# EnsembleRetriever (FAISS + Chroma)
#   ↓ retrieves 10 chunks
# EmbeddingsRedundantFilter
#   ↓ removes duplicates → 7 chunks
# EmbeddingsClusteringFilter
#   ↓ picks best per cluster → 4 chunks
# LongContextReorder
#   ↓ reorders for LLM attention → 4 chunks
# LLM gets clean, diverse, well-ordered context ✅

In [50]:
from models import get_model , get_embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableLambda

In [51]:
llm = get_model()

In [52]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context below.

Context: {context}
Question: {question}
""")

In [55]:
# First join contents into a single string
context_str = "\n\n".join(contents)

# Then pass directly
normal_chain = (
    {"context": RunnableLambda(lambda x: context_str), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [56]:
query = "what is current university of rohanta"

print(normal_chain.invoke(query))

The information provided does not mention the current university of Rohanta. It only mentions the universities that Rohanta has attended in the past, which are:

1. Savitribai Phule Pune University, Pune, India (for both Master of Engineering (ME) and Bachelor of Engineering (BE) degrees)
